## VISUALIZACIONES PARA EVALUAR LOS MODELOS

POSIBLES MEJORAS:
- analisis numerico, no solo graficas
- https://plotly.com/python/ml-regression/
- https://facebookresearch.github.io/hiplot/ - visualizacion de redes neuronales
- visualizar tanto para el conjunto test como el train

In [146]:
import pandas as pd
df = pd.read_csv('../../eventos_espera_semana_hp.csv')
df.head()

,ICAO,llegada_punto,salida_punto,despegue,tiempo_espera,aircraft_type,llegada_lon,llegada_lat,salida_lon,salida_lat,holding_point,parado,runway,fecha_despegue,hora_despegue
0,39bdae,2024-12-07 10:01:23.317,2024-12-07 10:02:45.266,2024-12-07 10:02:46.296,82.979,Medium 2 (between 34000 kg to 136000 kg),-3.561209,40.499423,-3.559227,40.506278,Y1,False,18L/36R,2024-12-07,10
1,4d24f1,2024-12-01 16:33:52.843,2024-12-01 16:35:40.634,2024-12-01 16:35:43.103,110.260,Medium 2 (between 34000 kg to 136000 kg),-3.576221,40.491579,-3.574645,40.497584,Z2,False,18R/36L,2024-12-01,16
2,71c082,2024-12-07 20:53:30.784,2024-12-07 20:55:13.042,2024-12-07 20:55:13.148,102.364,Heavy (larger than 136000 kg),-3.576253,40.491532,-3.574631,40.496567,Z2,False,18R/36L,2024-12-07,20
3,06a140,2024-12-01 18:26:29.008,2024-12-01 18:27:27.916,2024-12-01 18:27:29.141,60.133,Heavy (larger than 136000 kg),-3.561240,40.499423,-3.559211,40.505358,Y1,False,18L/36R,2024-12-01,18
4,06a140,2024-12-02 16:23:27.259,2024-12-02 16:24:41.134,2024-12-02 16:24:43.186,75.927,Heavy (larger than 136000 kg),-3.561234,40.499428,-3.559196,40.504543,Y1,False,18L/36R,2024-12-02,16


Predicciones son un ruido normal, con el objetivo de probar los gráficos

In [147]:
import plotly.express as px
import plotly.graph_objects as go
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

df['prediccion_tiempo_espera'] = df['tiempo_espera'] + np.random.normal(0, 20, size=len(df))

# Evaluación del modelo
mse = mean_squared_error(df['tiempo_espera'], df['prediccion_tiempo_espera'])
rmse = mse ** 0.5
mae = mean_absolute_error(df['tiempo_espera'], df['prediccion_tiempo_espera'])
r2 = r2_score(df['tiempo_espera'], df['prediccion_tiempo_espera'])

print(f'MSE: {mse}')
print(f'MAE: {mae}')
print(f'RMSE: {rmse}')
print(f'R2: {r2}')

MSE: 396.72111609105485
MAE: 15.964446495709725
RMSE: 19.917859224601795
R2: 0.9999961438269942


### Distribución de los errores en el tiempo de espera

In [148]:
import plotly.express as px
import pandas as pd

# Suponiendo que df ya está cargado y contiene las columnas necesarias

# Crear la columna de error (predicción - valor real)
df['error'] = df['prediccion_tiempo_espera'] - df['tiempo_espera']

# Crear el histograma con agrupación por 'aircraft_type'
fig_error = px.histogram(df, 
                          x='error', 
                          nbins=100, 
                          title='Distribución del Error (Predicción - Real)', 
                          color='aircraft_type',  # Aquí añadimos el tipo de avión
                          labels={'error': 'Error (segundos)', 'aircraft_type': 'Tipo de Avión'},
                          color_discrete_map={
                              'Medium 2 (between 34000 kg to 136000 kg)': 'blue',
                              'Small (under 34000 kg)': 'green',
                              'Large (above 136000 kg)': 'red'
                          })

# Agregar anotaciones para indicar sobreestima e infraestima
fig_error.add_annotation(
    x=-df['error'].max(),
    y=10,
    text='Infraestima',
    showarrow=False,
    font=dict(size=12, color="black"),
    xanchor="center",
    yanchor="bottom"
)

fig_error.add_annotation(
    x=df['error'].max(),
    y=10,
    text='Sobreestima',
    showarrow=False,
    font=dict(size=12, color="black"),
    xanchor="center",
    yanchor="bottom"
)

# Agregar una línea vertical para x=0
fig_error.add_vline(
    x=0,
    line=dict(color="red", width=2, dash="dash"),
)

# Mostrar el gráfico
fig_error.show()


#### MEJORAS POSIBLES
- HISTOGRAMAS POR HORA, POR PUNTO DE ESPERA, PISTAS DE DESPEGUE...
- COMO REJILLA, O HACERLO PARECIDO AL ANTERIOR

In [158]:
import pandas as pd
import plotly.express as px

# Calcular el error
df['error'] = df['prediccion_tiempo_espera'] - df['tiempo_espera']
df['hora'] = pd.to_datetime(df['llegada_punto']).dt.hour

# Agrupar por hora y tipo de avión, calcular el error medio
error_mean = df.groupby(['aircraft_type', 'hora'])['error'].mean().reset_index()

# Pivot para el heatmap
heatmap_data = error_mean.pivot(index='aircraft_type', columns='hora', values='error')

# Crear el heatmap
fig = px.imshow(
    heatmap_data,
    color_continuous_scale='RdBu',
    origin='lower',
    labels=dict(x='Hora del Día', y='Tipo de Avión', color='Error Medio (s)'),
    aspect='auto',
    title='Error Medio (Predicción - Real) por Hora y Tipo de Avión',
)

# Asegurar ticks en el eje X cada hora
horas_disponibles = sorted(df['hora'].unique())
fig.update_xaxes(
    tickmode='array',
    tickvals=list(range(24)),
    ticktext=[f"{h:02d}" for h in range(24)],
    side='bottom'
)

# Ajustes generales
fig.update_layout(
    xaxis_title="Hora del Día",
    yaxis_title="Tipo de Avión",
    height=500,
    width=1000
)

fig.show()


In [168]:
import pandas as pd
import plotly.graph_objects as go

# Calcular columnas necesarias
df['error'] = df['prediccion_tiempo_espera'] - df['tiempo_espera']
df['hora'] = pd.to_datetime(df['llegada_punto']).dt.hour

# Calcular agrupaciones
agg_mean = df.groupby(['holding_point', 'hora'])['error'].mean().reset_index()
worstCol=lambda x: max(x.min(), x.max(), key=abs)
agg_worst = df.groupby(['holding_point', 'hora'])['error'].apply(worstCol).reset_index()
bestCol=lambda x: min(x.min(), x.max(), key=abs)
agg_best = df.groupby(['holding_point', 'hora'])['error'].apply(bestCol).reset_index()
agg_median = df.groupby(['holding_point', 'hora'])['error'].median().reset_index()

# Pivot para cada métrica
heatmap_mean = agg_mean.pivot(index='holding_point', columns='hora', values='error')
heatmap_worst = agg_worst.pivot(index='holding_point', columns='hora', values='error')
heatmap_best = agg_best.pivot(index='holding_point', columns='hora', values='error')
heatmap_median = agg_median.pivot(index='holding_point', columns='hora', values='error')

# Crear figura inicial con la media
fig = go.Figure()

# Añadir como trazas separadas (inicialmente solo visible la media)
fig.add_trace(go.Heatmap(
    z=heatmap_mean.values,
    x=heatmap_mean.columns,
    y=heatmap_mean.index,
    colorscale='RdBu',
    zmid=0,
    colorbar=dict(title='Error (s)'),
    visible=True,
    name='Media'
))

fig.add_trace(go.Heatmap(
    z=heatmap_median.values,
    x=heatmap_median.columns,
    y=heatmap_median.index,
    colorscale='RdBu',
    zmid=0,
    visible=False,
    showscale=False,
    name='Mediana'
))

fig.add_trace(go.Heatmap(
    z=heatmap_worst.values,
    x=heatmap_worst.columns,
    y=heatmap_worst.index,
    colorscale='RdBu',
    zmid=0,
    visible=False,
    showscale=False,
    name='Peor'
))

fig.add_trace(go.Heatmap(
    z=heatmap_best.values,
    x=heatmap_best.columns,
    y=heatmap_best.index,
    colorscale='RdBu',
    zmid=0,
    visible=False,
    showscale=False,
    name='Mejor'
))

# Selector de métrica
fig.update_layout(
    updatemenus=[
        {
            'buttons': [
                {
                    'label': 'Media',
                    'method': 'update',
                    'args': [{'visible': [True, False, False, False]},
                             {'title': 'Error Medio (Predicción - Real) por Hora y Punto de Espera'}]
                },
                {
                    'label': 'Mediana',
                    'method': 'update',
                    'args': [{'visible': [False, True, False, False]},
                             {'title': 'Error Mediana (Predicción - Real) por Hora y Punto de Espera'}]
                },
                {
                    'label': 'Peor',
                    'method': 'update',
                    'args': [{'visible': [False, False, True, False]},
                             {'title': 'Peor Error ABS(Predicción - Real) por Hora y Punto de Espera'}]
                },

                {
                    'label': 'Mejor',
                    'method': 'update',
                    'args': [{'visible': [False, False, False, True]},
                             {'title': 'Mejor Error ABS(Predicción - Real) por Hora y Punto de Espera'}]
                },
                
            ],
            'direction': 'right',
            'type': 'buttons',
            'showactive': True,
            'x': 0.5,
            'xanchor': 'center',
            'y': -0.2,
            'yanchor': 'top'
        }
    ]
)

# Ejes
fig.update_xaxes(
    tickmode='array',
    tickvals=list(range(24)),
    ticktext=[f"{h:02d}" for h in range(24)],
    title='Hora del Día'
)

fig.update_yaxes(title='Punto de Espera')

fig.update_layout(
    title='Error Medio (Predicción - Real) por Hora y Punto de Espera',
    height=600,
    width=1000,
    margin=dict(t=50, b=100)
)

fig.show()


In [149]:
import plotly.express as px
df_filtrado = df[df["tiempo_espera"] < 240]

# Crear el gráfico de dispersión
fig_scatter = px.scatter(df_filtrado, x='tiempo_espera', y='prediccion_tiempo_espera', 
                         color='aircraft_type',  # Colorear los puntos por tipo de aeronave
                         title='Real vs Predicción', 
                         labels={'tiempo_espera': 'Tiempo Espera Real', 'prediccion_tiempo_espera': 'Predicción Tiempo Espera'})

xy_line = {
    'type': "line",
    'x0': df_filtrado['tiempo_espera'].min(),
    'y0': df_filtrado['tiempo_espera'].min(),
    'x1': df_filtrado['tiempo_espera'].max(),
    'y1': df_filtrado['tiempo_espera'].max(),
    'line': {'color': "black", 'width': 2, 'dash': 'dash'}
}

fig_scatter.add_shape(xy_line)

# Añadir las anotaciones "Sobreestima" e "Infraestima"
fig_scatter.add_annotation(
    x=df_filtrado['tiempo_espera'].min(), y=df_filtrado['prediccion_tiempo_espera'].max(), 
    text="Sobreestima", 
    showarrow=False, 
    font=dict(size=12, color="black"), 
    xanchor="left", 
    yanchor="top"
)

fig_scatter.add_annotation(
    x=df_filtrado['tiempo_espera'].max(), y=df_filtrado['prediccion_tiempo_espera'].min(), 
    text="Infraestima", 
    showarrow=False, 
    font=dict(size=12, color="black"), 
    xanchor="right", 
    yanchor="bottom"
)
min_val = 0
max_val = 240
tick_vals = np.linspace(int(min_val), int(max_val), num=17)

fig_scatter.update_layout(
    autosize=False,
    width=900,
    height=600,
    margin=dict(t=120, b=60, l=60, r=60),
    xaxis=dict(
        range=[min_val, max_val],
        tickvals=tick_vals,
    ),
    yaxis=dict(
        range=[min_val, max_val],
        tickvals=tick_vals
    ),
    updatemenus=[
        dict(
            type="buttons",
            direction="right",
            showactive=True,
            xanchor="center",
            yanchor="top",
            x=0.5,
            y=-0.2,
            pad={"r": 0, "t": 0},
            buttons=[
                dict(
                    label="Show line (X=Y)",
                    method="relayout",
                    args=[{"shapes": [xy_line]}]
                ),
                dict(
                    label="Hide",
                    method="relayout",
                    args=[{"shapes": []}]
                )
            ]
        )
    ],
)

# Mostrar el gráfico
fig_scatter.show()


In [205]:
import plotly.express as px
import pandas as pd

# Umbrales
umbral = 5  # segundos

# Calcular error
df['error'] = df['tiempo_espera'] - df['prediccion_tiempo_espera']

# Clasificar errores
def clasificar_error(e):
    if e < -umbral:
        return 'Infraestima'
    elif e > umbral:
        return 'Sobreestima'
    else:
        return 'Buena predicción'

df['clase_error'] = df['error'].apply(clasificar_error)

# Calcular tamaño del punto (más grande si se aleja del umbral)
def calcular_tamaño(error):
    if error < -umbral:
        return abs(error + umbral) / 1000
    elif error > umbral:
        return abs(error - umbral) / 1000
    else:
        return 0.01  # punto mínimo

df['tamaño_punto'] = df['error'].apply(calcular_tamaño)

# Crear el mapa
fig = px.scatter_mapbox(
    df,
    lat="salida_lat",
    lon="salida_lon",
    color="clase_error",
    size="tamaño_punto",
    size_max=10,  # tamaño máximo razonable
    color_discrete_map={
        "Infraestima": "blue",
        "Buena predicción": "green",
        "Sobreestima": "red"
    },
    hover_name="ICAO",
    hover_data={
        "aircraft_type": True,
        "error": ':.2f',
        "clase_error": True,
        "salida_lat": False,
        "salida_lon": False
    },
    zoom=9,
    mapbox_style="carto-positron"
)

fig.update_traces(marker=dict(opacity=0.75))

fig.update_layout(
    title="Errores de Predicción sobre Mapa (punto mas grande - mayor error) coordenandas = salida del pt. espera",
    legend_title_text='Clase de Error',
    margin=dict(t=40, b=0, l=0, r=0)
)

fig.show()


/var/folders/d9/w858q_3s5xj1lvw2flnmb_jr0000gn/T/ipykernel_34340/843266676.py:33: DeprecationWarning:

*scatter_mapbox* is deprecated! Use *scatter_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/



In [151]:
df[df['error'] > 5]['error'].max()

67.64762686966206

In [152]:
df[df['error'] < -5]['error'].min()

-67.53972154923659

In [203]:
import plotly.express as px
import pandas as pd

# Umbrales
umbral = 5  # segundos

# Calcular error
df['error'] = df['tiempo_espera'] - df['prediccion_tiempo_espera']

# Clasificar errores
def clasificar_error(e):
    if e < -umbral:
        return 'Infraestima'
    elif e > umbral:
        return 'Sobreestima'
    else:
        return 'Buena predicción'

df['clase_error'] = df['error'].apply(clasificar_error)

# Calcular opacidad: entre 0.2 (bajo) y 1.0 (alto)
def calcular_opacidad(error):
    if error < -umbral:
        return (abs(error) / 70)
    elif error > umbral:
        return (error / 78)
    else:
        return 1

df['opacidad'] = df['error'].apply(calcular_opacidad)

# Crear figura base sin pasar tamaño
fig = px.scatter_mapbox(
    df,
    lat="salida_lat",
    lon="salida_lon",
    color="clase_error",
    color_discrete_map={
        "Infraestima": "blue",
        "Buena predicción": "green",
        "Sobreestima": "red"
    },
    hover_name="ICAO",
    hover_data={
        "aircraft_type": True,
        "error": ':.2f',
        "clase_error": True,
        "opacidad": False,
        "salida_lat": False,
        "salida_lon": False
    },
    zoom=9,
    mapbox_style="carto-positron"
)

# Asignar opacidad punto por punto
fig.update_traces(marker=dict(size=9), selector=dict(mode='markers'))
for i, opacity in enumerate(df['opacidad']):
    fig.data[0].marker.opacity = df['opacidad']  # usar vector completo de opacidades

# Ajustes finales
fig.update_layout(
    title="Errores de Predicción sobre Mapa (mas opaco - mayor error) coordenandas = salida del pt. espera",
    legend_title_text='Clase de Error',
    margin=dict(t=40, b=0, l=0, r=0)
)

fig.show()


/var/folders/d9/w858q_3s5xj1lvw2flnmb_jr0000gn/T/ipykernel_34340/255450544.py:33: DeprecationWarning:

*scatter_mapbox* is deprecated! Use *scatter_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/



POSIBLES MEJORAS
- Añadir un mapa con el punto de espera y la media/mediana (se pueden meter botones como en el heatmap) del error que tiene


In [213]:
import pandas as pd
import plotly.express as px

# Asegúrate de tener la columna de error
df['error'] = df['prediccion_tiempo_espera'] - df['tiempo_espera']

# Agrupar los despegues por minuto


# Scatter plot
fig = px.scatter(
    df,
    x='despegue',
    y='error',
    color='aircraft_type',
    hover_data=['ICAO', 'error', 'aircraft_type', 'holding_point'],
    title='Errores de Predicción por Momento de Despegue<br>(para ver si hay algun problema por ser festivo...)',
    labels={'despegue': 'Despegue', 'error': 'Error (s)'}
)

fig.update_layout(
    xaxis_title='Momento de despegue',
    yaxis_title='Error de predicción (segundos)',
    height=600
)

fig.show()


POSIBLES MEJORAS
- Se puede de alguna forma representar el error dependiendo del número de aviones en puntos de espera
- En algunas graficas los tiempos de despegue saldrán duplicados, se podría eliminar si eliminamos duplicados sin la columna holding_point